# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
json_md = metadata.to_json()
print(f"{json_md['name']}: {json_md['description']}")
print(f"Dataset citation: {json_md.get('citeAs', json_md.get('citation', ''))}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# Explore record sets and fields
record_sets = []

if hasattr(metadata, 'record_sets') and metadata.record_sets:
    for record_set in metadata.record_sets:
        print(f"RecordSet: {record_set.id} -- {record_set.name}")
        record_sets.append(record_set.id)
        print("  Fields:")
        for field in record_set.fields:
            print(f"    Field: {field.id} -- {field.name} (dataType: {field.data_type})")
else:
    print("No record sets found in metadata. Attempting to extract from the distribution...")
    # Try to explore via distribution objects
    if hasattr(metadata, 'distributions'):
        for d in metadata.distributions:
            print(f"Distribution: {getattr(d, 'id', 'N/A')} - {getattr(d, 'name', 'N/A')}")
    else:
        print("No distributions found.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
dataframes = {}
# We'll try with the first record set, or fallback to distribution if no record sets
if record_sets:
    selected_record_set = record_sets[0]
    print(f"Extracting from RecordSet: {selected_record_set}")
    records = list(dataset.records(record_set=selected_record_set))
    df = pd.DataFrame(records)
    dataframes[selected_record_set] = df
    print("Fields/columns in the DataFrame:")
    print(df.columns.tolist())
    display(df.head())
else:
    # Try to instantiate DataFrame directly from records
    print("No record sets available directly. Attempting to iterate dataset.records() without argument...")
    raw_records = list(dataset.records())
    df = pd.DataFrame(raw_records)
    dataframes['main'] = df
    print("Fields/columns in the DataFrame:")
    print(df.columns.tolist())
    display(df.head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

In [ ]:
import numpy as np

# Identify a numeric field for demonstration, fallback to common ones in proteomics
if 'main' in dataframes:
    df = dataframes['main']
else:
    df = dataframes[selected_record_set]

print("Available columns: ", df.columns.tolist())

# Try finding a likely numeric field (e.g., 'MW', 'Coverage', 'PeptideCount')
numeric_fields = [c for c in df.columns if df[c].dtype in [np.float64, np.int64] or pd.api.types.is_numeric_dtype(df[c])]
if not numeric_fields:
    # Try to coerce possible proteomics fields
    for nf in ['Coverage', 'MW', 'PeptideCount', 'Spectra', 'Normalized_Abundance', 'Molecular_Weight']:
        if nf in df.columns:
            df[nf] = pd.to_numeric(df[nf], errors='coerce')
            numeric_fields.append(nf)
numeric_field = numeric_fields[0] if numeric_fields else df.select_dtypes(include=np.number).columns[0]
print(f"Selected numeric field: {numeric_field}")

threshold = df[numeric_field].quantile(0.75) # use 75th percentile as example threshold
filtered_df = df[df[numeric_field] > threshold].copy()
print(f"Filtered records with {numeric_field} > {threshold}:")
display(filtered_df.head(10))

filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"Normalized {numeric_field} for filtered records:")
display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head(10))

# Attempt group by a likely categorical field -- 'Description' or similar.
possible_group_fields = ['Description', 'Protein', 'Sample', 'Accession', 'Gene']
group_field = None
for g in possible_group_fields:
    if g in df.columns:
        group_field = g
        break
if group_field:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
    print(f"Grouped data by {group_field} (showing mean {numeric_field}):")
    display(grouped_df.head())
else:
    print("No suitable group field found in columns.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8,4))
sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
plt.title(f'Distribution of {numeric_field}')
plt.xlabel(numeric_field)
plt.ylabel('Frequency')
plt.show()

if group_field:
    plt.figure(figsize=(10,6))
    sns.boxplot(x=group_field, y=numeric_field, data=filtered_df)
    plt.title(f'{numeric_field} by {group_field}')
    plt.xticks(rotation=90)
    plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.


- We loaded the Croissant-formatted proteomics dataset using the `mlcroissant` library.
- We programmatically identified record sets and available fields/columns referenced by their `@id`s (when available).
- Numeric fields (e.g., Molecular Weight, Coverage, etc.) were filtered and normalized to demonstrate typical EDA steps.
- We visualized data distributions to gain insight into the variation and potential outliers.

Further work may include:
- Domain-specific filtering/grouping (e.g., by sample type, protein family)
- More advanced visualizations, such as hierarchical clustering, heatmaps, or volcano plots
- Exporting processed data for downstream bioinformatics or machine learning applications.